In [0]:
df = spark.read.json("/Volumes/workspace/default/appliances_review_dataset/appliances_review.jsonl")

In [0]:
df.printSchema()


root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [0]:
from pyspark.sql.functions import concat_ws
df = df.withColumn(
    "combined_text",
    concat_ws(" ", df.title, df.text)
)


In [0]:
df.select("title", "text", "combined_text").show(5, truncate=False)

+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|title            |text                                                                                                                                                                                                                                                                                                       |combined_text                                  

In [0]:
%pip install emoji

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import emoji
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType


In [0]:
def emoji_to_text(text):
    if text is None:
        return None
    return emoji.demojize(text, delimiters=(" ", " "))


In [0]:
emoji_udf = udf(emoji_to_text, StringType())

In [0]:
df = df.withColumn(
    "text_emoji",
    emoji_udf("combined_text")
)


In [0]:
df.select("combined_text", "text_emoji").show(5, truncate=False)


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|combined_text                                                                                                                                                                                                                                                                                                            |text_emoji                                             

In [0]:
from pyspark.sql.functions import col, regexp_replace, lower, trim


In [0]:
df = df.withColumn(
    "english_text",
    lower(
        trim(
            regexp_replace(
                col("text_emoji"),
                "[^a-zA-Z ]",
                ""
            )
        )
    )
)


In [0]:
df.select("text_emoji", "english_text").show(5, truncate=False)


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|text_emoji                                                                                                                                                                                                                                                                                                               |english_text                                                       

In [0]:
from pyspark.ml.feature import Tokenizer


In [0]:
tokenizer = Tokenizer(
    inputCol="english_text",
    outputCol="words"
)


In [0]:
df = tokenizer.transform(df)


In [0]:
from pyspark.ml.feature import StopWordsRemover


In [0]:
stopword_remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)


In [0]:
df = stopword_remover.transform(df)


In [0]:
from pyspark.sql.functions import concat_ws


In [0]:
df = df.withColumn(
    "final_text",
    concat_ws(" ", df.filtered_words)
)


In [0]:
# display(df.select("final_text"))
df.select(
    "combined_text",
    "final_text"
).show(5, truncate=False)


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|combined_text                                                                                                                                                                                                                                                                                                            |final_text                                                                                                                                                                          |
+-----

In [0]:
df.filter(
    df.combined_text != df.text_emoji
).select(
    "combined_text",
    "text_emoji"
).show(20, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
df.filter(df.combined_text != df.text_emoji).count()

22819

### FILTERING AND CLEANING ENDS HERE 

### **NOW WE ARE JUST CHECKING THE DATA**

In [0]:
display(
    df.select(
        "title",
        "text",
        "combined_text",
        "text_emoji",
        "english_text",
        "final_text"
    )
)


title,text,combined_text,text_emoji,english_text,final_text
Work great,work great. use a new one every month,Work great work great. use a new one every month,Work great work great. use a new one every month,work great work great use a new one every month,work great work great use new one every month
excellent product,Little on the thin side,excellent product Little on the thin side,excellent product Little on the thin side,excellent product little on the thin side,excellent product little thin side
Happy customer!,"Quick delivery, fixed the issue!","Happy customer! Quick delivery, fixed the issue!","Happy customer! Quick delivery, fixed the issue!",happy customer quick delivery fixed the issue,happy customer quick delivery fixed issue
Amazing value,"I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.","Amazing value I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.","Amazing value I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.",amazing value i wasnt sure whether these were worth it or not given the cost compared to the original branded filtersbr br i can happily report that these are a great value and work every bit as good as the original if you are on the fence worrying whether these are worth it i can assure you they are,amazing value wasnt sure whether worth given cost compared original branded filtersbr br happily report great value work every bit good original fence worrying whether worth assure
Dryer parts,Easy to install got the product expected to receive,Dryer parts Easy to install got the product expected to receive,Dryer parts Easy to install got the product expected to receive,dryer parts easy to install got the product expected to receive,dryer parts easy install got product expected receive
DO NOT purchase this ice machine.,After buying this ice machine just 15 months ago and using it 5 times per month it’s now leaking so bad I can’t use it anymore. The company has refused to replace it!,DO NOT purchase this ice machine. After buying this ice machine just 15 months ago and using it 5 times per month it’s now leaking so bad I can’t use it anymore. The company has refused to replace it!,DO NOT purchase this ice machine. After buying this ice machine just 15 months ago and using it 5 times per month it’s now leaking so bad I can’t use it anymore. The company has refused to replace it!,do not purchase this ice machine after buying this ice machine just months ago and using it times per month its now leaking so bad i cant use it anymore the company has refused to replace it,purchase ice machine buying ice machine months ago using times per month leaking bad cant use anymore company refused replace
They don't fit properly,Not the best quality,They don't fit properly Not the best quality,They don't fit properly Not the best quality,they dont fit properly not the best quality,dont fit properly best quality
Five Stars,Part came quickly and fit my LG dryer. Thanks!,Five Stars Part came quickly and fit my LG dryer. Thanks!,Five Stars Part came quickly and fit my LG dryer. Thanks!,five stars part came quickly and fit my lg dryer thanks,five stars part came quickly fit lg dryer thanks
Five Stars,Always arrive in a fast manner. Descriptions on line are accurate and helpful,Five Stars Always arrive in a fast manner. Descripti

In [0]:
display(
    df.filter(
        (df.combined_text != df.text_emoji) |
        (df.text_emoji != df.english_text) |
        (df.english_text != df.final_text)
    ).select(
        "combined_text",
        "text_emoji",
        "english_text",
        "final_text"
    ).limit(20)
)


combined_text,text_emoji,english_text,final_text
Work great work great. use a new one every month,Work great work great. use a new one every month,work great work great use a new one every month,work great work great use new one every month
excellent product Little on the thin side,excellent product Little on the thin side,excellent product little on the thin side,excellent product little thin side
"Happy customer! Quick delivery, fixed the issue!","Happy customer! Quick delivery, fixed the issue!",happy customer quick delivery fixed the issue,happy customer quick delivery fixed issue
"Amazing value I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.","Amazing value I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.",amazing value i wasnt sure whether these were worth it or not given the cost compared to the original branded filtersbr br i can happily report that these are a great value and work every bit as good as the original if you are on the fence worrying whether these are worth it i can assure you they are,amazing value wasnt sure whether worth given cost compared original branded filtersbr br happily report great value work every bit good original fence worrying whether worth assure
Dryer parts Easy to install got the product expected to receive,Dryer parts Easy to install got the product expected to receive,dryer parts easy to install got the product expected to receive,dryer parts easy install got product expected receive
DO NOT purchase this ice machine. After buying this ice machine just 15 months ago and using it 5 times per month it’s now leaking so bad I can’t use it anymore. The company has refused to replace it!,DO NOT purchase this ice machine. After buying this ice machine just 15 months ago and using it 5 times per month it’s now leaking so bad I can’t use it anymore. The company has refused to replace it!,do not purchase this ice machine after buying this ice machine just months ago and using it times per month its now leaking so bad i cant use it anymore the company has refused to replace it,purchase ice machine buying ice machine months ago using times per month leaking bad cant use anymore company refused replace
They don't fit properly Not the best quality,They don't fit properly Not the best quality,they dont fit properly not the best quality,dont fit properly best quality
Five Stars Part came quickly and fit my LG dryer. Thanks!,Five Stars Part came quickly and fit my LG dryer. Thanks!,five stars part came quickly and fit my lg dryer thanks,five stars part came quickly fit lg dryer thanks
Five Stars Always arrive in a fast manner. Descriptions on line are accurate and helpful,Five Stars Always arrive in a fast manner. Descriptions on line are accurate and helpful,five stars always arrive in a fast manner descriptions on line are accurate and helpful,five stars always arrive fast manner descriptions line accurate helpful
Company is phenomenal. The company responded very quickly. Refunded purchase price right away. No problems at all. The unit did not work. I tried ev,Company is phenomenal. The company responded very quickly. Refunded purchase price right away. No problems at all. The unit did not work. I tried ev,company is phenomenal the company responded very quickly refunded purchase price right away no problems at all the unit did not work i tried ev,company phenomenal company responded quickly refunded purchase price right away problems unit work tried ev


In [0]:
display(df)


asin,helpful_vote,parent_asin,rating,text,timestamp,title,user_id,verified_purchase,combined_text,text_emoji,english_text,words,filtered_words,final_text
B01N0TQ0OH,0,B01N0TQ0OH,5.0,work great. use a new one every month,1519317108692,Work great,AGKHLEW2SOWHNMFQIJGBECAF7INQ,true,Work great work great. use a new one every month,Work great work great. use a new one every month,work great work great use a new one every month,"List(work, great, work, great, use, a, new, one, every, month)","List(work, great, work, great, use, new, one, every, month)",work great work great use new one every month
B07DD2DMXB,0,B07DD37QPZ,5.0,Little on the thin side,1664746863446,excellent product,AHWWLSPCJMALVHDDVSUGICL6RUCA,true,excellent product Little on the thin side,excellent product Little on the thin side,excellent product little on the thin side,"List(excellent, product, little, on, the, thin, side)","List(excellent, product, little, thin, side)",excellent product little thin side
B082W3Z9YK,0,B082W3Z9YK,5.0,"Quick delivery, fixed the issue!",1607225435363,Happy customer!,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,true,"Happy customer! Quick delivery, fixed the issue!","Happy customer! Quick delivery, fixed the issue!",happy customer quick delivery fixed the issue,"List(happy, customer, quick, delivery, fixed, the, issue)","List(happy, customer, quick, delivery, fixed, issue)",happy customer quick delivery fixed issue
B078W2BJY8,0,B078W2BJY8,5.0,"I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.",1534104184306,Amazing value,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,true,"Amazing value I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.","Amazing value I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.",amazing value i wasnt sure whether these were worth it or not given the cost compared to the original branded filtersbr br i can happily report that these are a great value and work every bit as good as the original if you are on the fence worrying whether these are worth it i can assure you they are,"List(amazing, value, i, wasnt, sure, whether, these, were, worth, it, or, not, given, the, cost, compared, to, the, original, branded, filtersbr, br, i, can, happily, report, that, these, are, a, great, value, and, work, every, bit, as, good, as, the, original, if, you, are, on, the, fence, worrying, whether, these, are, worth, it, i, can, assure, you, they, are)","List(amazing, value, wasnt, sure, whether, worth, given, cost, compared, original, branded, filtersbr, br, happily, report, great, value, work, every, bit, good, original, fence, worrying, whether, worth, assure)",amazing value wasnt sure whether worth given cost compared original branded filtersbr br happily report great value work every bit good original fence worrying whether worth assure
B08C9LPCQV,0,B08C9LPCQV,5.0,Easy to install got the product expected to receive,1620176603754,Dryer parts,AELFJFAXQERUSMTXJQ6SYFFRDWMA,true,Dryer parts Easy to install got the product expected to receive,Dryer parts Easy to install got the product expected to receive,dryer parts easy to install got the product expected to receive,"List(dryer, parts, easy, to, install, got, the, product, expected, to, receive)","List(dryer, parts, easy, install, got, product, expected, receive)",dryer parts easy install got product expected recei

In [0]:
display(df.limit(20))


asin,helpful_vote,parent_asin,rating,text,timestamp,title,user_id,verified_purchase,combined_text,text_emoji,english_text,words,filtered_words,final_text
B01N0TQ0OH,0,B01N0TQ0OH,5.0,work great. use a new one every month,1519317108692,Work great,AGKHLEW2SOWHNMFQIJGBECAF7INQ,true,Work great work great. use a new one every month,Work great work great. use a new one every month,work great work great use a new one every month,"List(work, great, work, great, use, a, new, one, every, month)","List(work, great, work, great, use, new, one, every, month)",work great work great use new one every month
B07DD2DMXB,0,B07DD37QPZ,5.0,Little on the thin side,1664746863446,excellent product,AHWWLSPCJMALVHDDVSUGICL6RUCA,true,excellent product Little on the thin side,excellent product Little on the thin side,excellent product little on the thin side,"List(excellent, product, little, on, the, thin, side)","List(excellent, product, little, thin, side)",excellent product little thin side
B082W3Z9YK,0,B082W3Z9YK,5.0,"Quick delivery, fixed the issue!",1607225435363,Happy customer!,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,true,"Happy customer! Quick delivery, fixed the issue!","Happy customer! Quick delivery, fixed the issue!",happy customer quick delivery fixed the issue,"List(happy, customer, quick, delivery, fixed, the, issue)","List(happy, customer, quick, delivery, fixed, issue)",happy customer quick delivery fixed issue
B078W2BJY8,0,B078W2BJY8,5.0,"I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.",1534104184306,Amazing value,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,true,"Amazing value I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.","Amazing value I wasn't sure whether these were worth it or not, given the cost compared to the original branded filters.I can happily report that these are a great value and work every bit as good as the original. If you are on the fence worrying whether these are worth it- I can assure you they are.",amazing value i wasnt sure whether these were worth it or not given the cost compared to the original branded filtersbr br i can happily report that these are a great value and work every bit as good as the original if you are on the fence worrying whether these are worth it i can assure you they are,"List(amazing, value, i, wasnt, sure, whether, these, were, worth, it, or, not, given, the, cost, compared, to, the, original, branded, filtersbr, br, i, can, happily, report, that, these, are, a, great, value, and, work, every, bit, as, good, as, the, original, if, you, are, on, the, fence, worrying, whether, these, are, worth, it, i, can, assure, you, they, are)","List(amazing, value, wasnt, sure, whether, worth, given, cost, compared, original, branded, filtersbr, br, happily, report, great, value, work, every, bit, good, original, fence, worrying, whether, worth, assure)",amazing value wasnt sure whether worth given cost compared original branded filtersbr br happily report great value work every bit good original fence worrying whether worth assure
B08C9LPCQV,0,B08C9LPCQV,5.0,Easy to install got the product expected to receive,1620176603754,Dryer parts,AELFJFAXQERUSMTXJQ6SYFFRDWMA,true,Dryer parts Easy to install got the product expected to receive,Dryer parts Easy to install got the product expected to receive,dryer parts easy to install got the product expected to receive,"List(dryer, parts, easy, to, install, got, the, product, expected, to, receive)","List(dryer, parts, easy, install, got, product, expected, receive)",dryer parts easy install got product expected recei